# Inspect lexicon, collocation skeleton & GijsBERT dev

Regenerate this notebook — do not hand-edit JSON:

```bash
uv run python scripts/generate_notebooks.py --name inspect_lexicon_csvs
```

## Workflows

1. **Lexicon CSV** — review `keep`, noise terms (`oncen`, `ponden`, …), confirm `FrameVerbLexicon` loading.
2. **Collocation skeleton** — stream-search PMI collocates in `eval/collocation_skeleton.csv`.
3. **Dev skeleton audit** — hand-gold dev targets vs corpus-wide verb collocates (GijsBERT export go/no-go).

Paths use `data_io.resolve` only (see `data_manifest.toml`).


## Setup (manifest paths)

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

from data_io import resolve
from trifecta_annotation.frame_verbs import get_frame_verb_lexicon
from trifecta_annotation.normalize import normalize_hist_dutch


def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / 'trifecta_frame_verb_corpus.csv').exists():
            return p
    return cwd


REPO_ROOT = _find_repo_root()
GIJSBERT_DIR = Path(resolve('trifecta_gijsbert'))
EVAL_DIR = Path(resolve('eval_reports'))

LEXICON_CSV = REPO_ROOT / 'trifecta_frame_verb_corpus.csv'
SKELETON_CSV = EVAL_DIR / 'collocation_skeleton.csv'
DEV_JSONL = GIJSBERT_DIR / 'dev.jsonl'

for label, path in [
    ('lexicon', LEXICON_CSV),
    ('skeleton', SKELETON_CSV),
    ('dev', DEV_JSONL),
]:
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
    print(label, path)


## 1. Frame-verb lexicon CSV

Repo-root review file. Only `keep=yes` rows load into the merged lexicon (plus guideline/manual/technique).


In [ ]:
lex_df = pd.read_csv(LEXICON_CSV, dtype=str, keep_default_na=False)
print('lexicon rows:', len(lex_df))
lex_df.head(10)


In [ ]:
def show_keep_status(df: pd.DataFrame) -> dict[str, int]:
    keep = df['keep'].astype(str).str.strip().str.lower()
    return {
        'keep_yes': int(keep.isin({'yes', 'y', '1', 'true'}).sum()),
        'keep_blank': int((keep == '').sum()),
        'keep_no': int(keep.isin({'no', 'n', '0', 'false'}).sum()),
    }


show_keep_status(lex_df)


In [ ]:
def lookup_lexicon_terms(terms: list[str]) -> pd.DataFrame:
    terms_norm = pd.Series(terms, dtype='string').str.strip().str.lower()
    hit = lex_df[lex_df['term_norm'].astype(str).str.lower().isin(terms_norm)]
    return hit.sort_values('term_norm')


terms_to_check = ['oncen', 'ponden', 'sullen', 'binnen', 'dingen', 'hoemen', 'breken']
lookup_lexicon_terms(terms_to_check)


### Loaded triggers (`FrameVerbLexicon`)

Confirms runtime lexicon after CSV edits (`reload=True`).


In [ ]:
lex = get_frame_verb_lexicon(reload=True)

terms_to_check = [
    'oncen', 'ponden', 'sullen', 'binnen', 'dingen', 'hoemen',
    'breken', 'stampen', 'malen',
]

load_rows = pd.DataFrame(
    {
        'term': terms_to_check,
        'in_lexicon': [lex.entry_for(t) is not None for t in terms_to_check],
        'frame': [(lex.entry_for(t).frame.value if lex.entry_for(t) else '') for t in terms_to_check],
        'sources': [(sorted(lex.entry_for(t).sources) if lex.entry_for(t) else []) for t in terms_to_check],
    }
)
load_rows


## 2. Collocation skeleton (stream search)

Large PMI table at `eval_reports/collocation_skeleton.csv`. Diagnostic only — does not control lexicon loading.


In [ ]:
_SKELETON_USECOLS = [
    'target_norm',
    'collocate_norm',
    'collocate_type',
    'pmi',
    'frame_hint',
    'in_frame_lexicon',
    'cooc_count',
    'target_snippet_count',
    'rank_within_target',
]


def search_skeleton(
    target_norm: str | None = None,
    collocate_norm: str | None = None,
    collocate_type: str | None = None,
    topn: int = 25,
    chunksize: int = 200_000,
) -> pd.DataFrame:
    target_norm = str(target_norm).strip().lower() if target_norm else None
    collocate_norm = str(collocate_norm).strip().lower() if collocate_norm else None
    collocate_type = str(collocate_type).strip().lower() if collocate_type else None

    hits: list[pd.DataFrame] = []
    for chunk in pd.read_csv(
        SKELETON_CSV,
        dtype=str,
        usecols=_SKELETON_USECOLS,
        chunksize=chunksize,
        keep_default_na=False,
    ):
        mask = pd.Series(True, index=chunk.index)
        if target_norm:
            mask &= chunk['target_norm'].astype(str).str.lower().eq(target_norm)
        if collocate_norm:
            mask &= chunk['collocate_norm'].astype(str).str.lower().eq(collocate_norm)
        if collocate_type:
            mask &= chunk['collocate_type'].astype(str).str.lower().eq(collocate_type)
        sub = chunk[mask]
        if not sub.empty:
            hits.append(sub)
        if sum(len(x) for x in hits) >= topn:
            break

    if not hits:
        return pd.DataFrame(columns=_SKELETON_USECOLS)

    out = pd.concat(hits, ignore_index=True)
    out['pmi_num'] = pd.to_numeric(out['pmi'], errors='coerce').fillna(-1)
    out = out.sort_values(['pmi_num', 'rank_within_target'], ascending=[False, True])
    return out.drop(columns=['pmi_num']).head(topn)


search_skeleton(target_norm='bier', collocate_type='verb', topn=15)


In [ ]:
search_skeleton(collocate_norm='ponden', topn=20)
search_skeleton(collocate_norm='oncen', topn=20)
search_skeleton(target_norm='bier', collocate_norm='brouwen', topn=10)


## 3. GijsBERT dev gold + skeleton audit

Hand-gold dev split (`trifecta_gijsbert/dev.jsonl`). Compare **gold frame** vs **corpus-wide** top verb collocates per target.

Skeleton = target-level prior from all `food_snippets_long` rows — not the verb in this specific snippet.

**Go/no-go for wiring skeleton into `export_gijsbert.py`:**
- **Coverage** — framed dev targets with ≥1 skeleton verb
- **Hint alignment** — top skeleton `frame_hint` matches gold (when hint present)


In [ ]:
def _marked_verb(text: str) -> str:
    m = re.search(r'\[VRB\](.*?)\[/VRB\]', str(text), flags=re.DOTALL)
    return m.group(1).strip() if m else ''


dev = [json.loads(line) for line in DEV_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
dev_df = pd.DataFrame(dev)
dev_df['target_norm'] = dev_df['target_word'].map(normalize_hist_dutch)
dev_df['marked_verb'] = dev_df['text'].map(_marked_verb)
framed = dev_df[dev_df['label'] != 'NONE'].copy()

print('dev rows:', len(dev_df))
print('NONE:', int((dev_df['label'] == 'NONE').sum()))
print('framed:', len(framed))
print('with [VRB] mark:', int((dev_df['marked_verb'] != '').sum()))
framed[['record_id', 'target_norm', 'label', 'marked_verb']].head(12)


In [ ]:
def skeleton_verbs_for_targets(
    target_norms: list[str],
    *,
    topn_per_target: int = 3,
    chunksize: int = 200_000,
) -> pd.DataFrame:
    """One pass: top verb collocates per target_norm."""
    wanted = {str(t).strip().lower() for t in target_norms if str(t).strip()}
    if not wanted:
        return pd.DataFrame(columns=_SKELETON_USECOLS)

    hits: list[pd.DataFrame] = []
    for chunk in pd.read_csv(
        SKELETON_CSV,
        dtype=str,
        usecols=_SKELETON_USECOLS,
        chunksize=chunksize,
        keep_default_na=False,
    ):
        sub = chunk[
            chunk['target_norm'].astype(str).str.lower().isin(wanted)
            & chunk['collocate_type'].astype(str).str.lower().eq('verb')
        ]
        if not sub.empty:
            hits.append(sub)

    if not hits:
        return pd.DataFrame(columns=_SKELETON_USECOLS)

    verbs = pd.concat(hits, ignore_index=True)
    verbs['rank_num'] = pd.to_numeric(verbs['rank_within_target'], errors='coerce').fillna(9999)
    verbs = verbs.sort_values(['target_norm', 'rank_num', 'pmi'], ascending=[True, True, False])
    return verbs.groupby('target_norm', as_index=False).head(topn_per_target).drop(columns=['rank_num'])


def dev_skeleton_audit(framed_df: pd.DataFrame, topn_per_target: int = 3) -> pd.DataFrame:
    targets = framed_df['target_norm'].drop_duplicates().tolist()
    skel = skeleton_verbs_for_targets(targets, topn_per_target=topn_per_target)

    audit = framed_df[
        ['record_id', 'target_norm', 'label', 'marked_verb', 'corpus']
    ].drop_duplicates(subset=['record_id'])

    if skel.empty:
        audit = audit.assign(
            skeleton_verb_rows=0,
            top_skeleton_verbs='',
            top_skeleton_hints='',
        )
    else:
        agg = (
            skel.groupby('target_norm', as_index=False)
            .agg(
                skeleton_verb_rows=('collocate_norm', 'count'),
                top_skeleton_verbs=('collocate_norm', lambda s: ', '.join(s.astype(str))),
                top_skeleton_hints=('frame_hint', lambda s: ', '.join(h for h in s.astype(str) if h)),
            )
        )
        audit = audit.merge(agg, on='target_norm', how='left')
        audit['skeleton_verb_rows'] = audit['skeleton_verb_rows'].fillna(0).astype(int)
        audit['top_skeleton_verbs'] = audit['top_skeleton_verbs'].fillna('')
        audit['top_skeleton_hints'] = audit['top_skeleton_hints'].fillna('')

    audit['has_skeleton_verbs'] = audit['skeleton_verb_rows'] > 0
    audit['hint_matches_gold'] = audit.apply(
        lambda r: r['label'] in str(r['top_skeleton_hints']).split(', ')
        if str(r['top_skeleton_hints']).strip()
        else False,
        axis=1,
    )
    return audit.sort_values(['label', 'target_norm', 'record_id'])


audit_df = dev_skeleton_audit(framed)
audit_df.head(20)


In [ ]:
def audit_summary(audit: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for label, grp in audit.groupby('label', sort=False):
        n = len(grp)
        covered = int(grp['has_skeleton_verbs'].sum())
        hinted = int((grp['top_skeleton_hints'] != '').sum())
        aligned = int(grp['hint_matches_gold'].sum())
        rows.append(
            {
                'gold_frame': label,
                'dev_rows': n,
                'targets_with_skeleton_verbs': covered,
                'coverage_pct': round(100 * covered / n, 1) if n else 0.0,
                'rows_with_hints': hinted,
                'hint_matches_gold': aligned,
                'alignment_pct': round(100 * aligned / hinted, 1) if hinted else 0.0,
            }
        )
    total = len(audit)
    covered_all = int(audit['has_skeleton_verbs'].sum())
    hinted_all = int((audit['top_skeleton_hints'] != '').sum())
    aligned_all = int(audit['hint_matches_gold'].sum())
    rows.append(
        {
            'gold_frame': 'ALL_FRAMED',
            'dev_rows': total,
            'targets_with_skeleton_verbs': covered_all,
            'coverage_pct': round(100 * covered_all / total, 1) if total else 0.0,
            'rows_with_hints': hinted_all,
            'hint_matches_gold': aligned_all,
            'alignment_pct': round(100 * aligned_all / hinted_all, 1) if hinted_all else 0.0,
        }
    )
    return pd.DataFrame(rows)


summary_df = audit_summary(audit_df)
summary_df


In [ ]:
# Drill into one gold frame
audit_df[audit_df['label'] == 'INGESTION'][[
    'target_norm', 'marked_verb', 'top_skeleton_verbs', 'top_skeleton_hints', 'hint_matches_gold'
]]


In [ ]:
# Mismatches: skeleton has hints but none match gold frame
audit_df[
    audit_df['top_skeleton_hints'].astype(str).str.strip().ne('')
    & ~audit_df['hint_matches_gold']
][['label', 'target_norm', 'marked_verb', 'top_skeleton_verbs', 'top_skeleton_hints']]


## Steering loop

1. Edit `trifecta_frame_verb_corpus.csv` (`keep` + notes).
2. `get_frame_verb_lexicon(reload=True)` — confirm loaded triggers.
3. Re-export skeleton when lexicon changes should refresh `frame_hint`:
   `uv run python scripts/export_collocation_skeleton.py --summary`
4. Re-export GijsBERT splits after gold/silver changes:
   `uv run python scripts/export_gijsbert.py ...`
5. Regenerate this notebook after generator edits.
